In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import  f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import LabelEncoder

In [ ]:
train_data = pd.read_csv('data/train_data.csv')
train_data

,Created,CancelTime,DepartureTime,BillID,TicketID,ReserveStatus,UserID,Male,Price,CouponDiscount,...,Domestic,VehicleType,VehicleClass,TripReason,Vehicle,Cancel,HashPassportNumber_p,HashEmail,BuyerMobile,NationalCode
0,2022-10-23 09:38:49.110,NaN,2022-11-02 23:59:00,39710203,1091777.0,5,122885.0,True,6600000.0,34425.0,...,1,NaN,False,Work,Plane,0,NaN,66c7f29e3b92f3b77e20830ac29e7758037a53d2238a5b...,764974891906,477368495
1,2022-08-15 14:51:43.160,NaN,2022-08-18 04:15:00,38689463,1070902.0,5,876925.0,True,9500000.0,0.0,...,1,NaN,False,Int,Plane,0,NaN,b24634843858a4175d03422aa9e7211ec3b9f3ce4c481c...,27479149496,15987669
2,2022-09-20 17:25:27.250,NaN,2022-09-21 11:00:00,39245173,7624237.0,3,916640.0,False,2000000.0,0.0,...,1,VIP 2+1,True,Work,Bus,0,NaN,NaN,323657282999,667640412
3,2022-06-25 11:32:53.980,NaN,2022-06-26 08:30:00,37957585,2867547.0,2,NaN,False,40000.0,0.0,...,1,3 ستاره اتوبوسي,NaN,Int,Train,0,NaN,NaN,169459057632,392476186
4,2022-06-01 11:30:53.633,NaN,2022-06-02 23:00:00,37584530,7212559.0,3,NaN,True,1130000.0,0.0,...,1,اسکانیا تک صندلی ۳۱نفره,True,Int,Bus,0,NaN,NaN,408595008421,79497837
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101012,2022-06-01 00:20:14.280,NaN,2022-06-04 12:10:00,37579327,1050781.0,5,NaN,True,5900000.0,0.0,...,1,بوئینگ 737,False,Int,Plane,0,NaN,NaN,605105141718,103215806
101013,2022-10-29 20:54:31.330,NaN,2022-11-01 15:30:00,39789479,3085407.0,2,403095.0,True,926500.0,0.0,...,1,4 ستاره اتوبوسي نگين,NaN,Work,Train,0,NaN,NaN,414997568556,193262890
101014,2022-09-03 17:57:22.067,NaN,2022-09-13 09:30:00,38991563,2322052.0,5,528307.0,True,30000000.0,0.0,...,0,NaN,False,Int,InternationalPlane,0,47b8f2d9b5de7e0e0e7234c18a1aa0c4b35798e6cb46b4...,a4dcb7941ee3c8f7b1fc6a171015692bc961d65a84ad47...,99460830937,34732401
101015,2022-09-29 13:15:51.303,NaN,2022-09-29 17:30:00,39406503,7664730.0,3,797946.0,True,980000.0,0.0,...,1,25 نفره (VIP),True,Work,Bus,0,NaN,718bc52c3e88520531463b385998a1193e2821d518b60b...,487489926847,458338866


In [ ]:
test_data = pd.read_csv('data/test_data.csv')
test_data

,Created,CancelTime,DepartureTime,BillID,TicketID,ReserveStatus,UserID,Male,Price,CouponDiscount,...,To,Domestic,VehicleType,VehicleClass,Vehicle,Cancel,HashPassportNumber_p,HashEmail,BuyerMobile,NationalCode
0,2022-07-22 12:52:35.293,NaN,2022-08-03 17:00:00,38372770,7429183.0,3,NaN,False,1220000.0,0.0,...,اصفهان,1,ماهانVIP مانیتوردار کاوه,True,Bus,0,NaN,7ec2bc45a1a56014c60beb9e6ae72e748563f270b5961e...,855578545022,752371627
1,2022-05-16 10:55:29.397,2022-05-18 11:35:15.643,2022-05-19 17:00:00,37348034,7147402.0,5,NaN,True,2830000.0,0.0,...,قم,1,VIP 2+1 / سیستم تهویه مطبوع / تخت شو,True,Bus,1,NaN,09731a27ab7323fa8a65a5d61a6f6cdf5b127a5d89d1af...,687891042993,926996211
2,2022-02-13 16:10:33.870,NaN,2022-02-15 22:45:00,36058084,6817410.0,3,NaN,True,1480000.0,0.0,...,اردبیل,1,25 نفره (VIP),True,Bus,0,NaN,NaN,515831406373,890191812
3,2022-05-01 17:56:18.530,NaN,2022-05-06 14:30:00,37140430,7093047.0,3,NaN,False,2450000.0,0.0,...,شیراز,1,Scania VIP 2+1,True,Bus,0,NaN,NaN,349742111820,941450742
4,2022-05-19 18:40:48.760,NaN,2022-05-21 12:30:00,37393459,2789874.0,2,135424.0,True,1965500.0,0.0,...,تهران,1,3 ستاره 6 تخته كوير,NaN,Train,0,NaN,a9b7344fa6dffcd85117f3b4209a379600a66019dbfb57...,63387630789,575541645
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43288,2022-10-06 16:57:16.053,NaN,2022-10-07 21:00:00,39515216,7693401.0,3,NaN,True,800000.0,0.0,...,زاهدان,1,classicus 2+2,True,Bus,0,NaN,084471d0e58161bf494c8d540208e17d2b3cb0ea3eff70...,793972140866,560495589
43289,2022-06-14 13:19:26.413,NaN,2022-06-18 07:15:00,37773311,1055789.0,5,NaN,False,7827000.0,0.0,...,تهران,1,NaN,False,Plane,0,NaN,NaN,574095530769,194295556
43290,2022-10-23 15:48:03.947,NaN,2022-10-23 20:25:00,39714891,7747506.0,3,NaN,False,370000.0,0.0,...,قم,1,مارال تک صندلی ۳۰نفره,True,Bus,0,NaN,NaN,82614935547,704290882
43291,2022-09-18 21:18:01.730,2022-09-19 15:40:44.707,2022-09-20 20:00:00,39215621,7620724.0,5,950877.0,True,2520000.0,0.0,...,مسجدسلیمان,1,VIP 2+1 / شارژر یو اس بی / سیستم تهویه مطبوع ...,True,Bus,1,NaN,57f1cffdba91d23d66f923c5defcd40caa8d317f8bcfbc...,242545550300,513968890


In [ ]:
y = train_data["TripReason"]

train = train_data.drop(columns=["TripReason"]).copy()
test  = test_data.copy()

DROP_COLS = [
    "UserID","BillID","TicketID",  
    "HashPassportNumber_p","HashEmail","BuyerMobile","NationalCode"
]

train = train.drop(columns=DROP_COLS, errors="ignore")
test  = test.drop(columns=DROP_COLS, errors="ignore")

def add_features(df):
    df = df.copy()

    df["Created"] = pd.to_datetime(df["Created"], errors="coerce")
    df["DepartureTime"] = pd.to_datetime(df["DepartureTime"], errors="coerce")

    df["days_to_departure"] = (df["DepartureTime"].dt.normalize() - df["Created"].dt.normalize()).dt.days
    df["days_to_departure"] = df["days_to_departure"].fillna(-1).astype(int)

    df = df.drop(columns=["Created", "DepartureTime", "CancelTime"], errors="ignore")

    for c in df.columns:
        if df[c].dtype == "bool":
            df[c] = df[c].astype("int8")

    return df

train = train.fillna("UNK")
test  = test.fillna("UNK")

train = add_features(train)
test  = add_features(test)


if "Vehicle" in train.columns and "Vehicle" in test.columns:
    all_vehicle = pd.concat([train["Vehicle"], test["Vehicle"]], axis=0).astype(str)
    codes, _ = pd.factorize(all_vehicle, sort=False)
    train["Vehicle_enc"] = codes[:len(train)]
    test["Vehicle_enc"]  = codes[len(train):]
    train = train.drop(columns=["Vehicle"])
    test  = test.drop(columns=["Vehicle"])


cols_to_drop = [c for c in ["UserID","BillID","TicketID","HashPassportNumber_p","HashEmail","BuyerMobile","NationalCode"] if c in train.columns]
train = train.drop(columns=cols_to_drop, errors="ignore")
test  = test.drop(columns=cols_to_drop, errors="ignore")


X = train
X_test = test

le = LabelEncoder()
y = le.fit_transform(y)

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]


X[cat_cols] = X[cat_cols].astype(str)
X_test[cat_cols] = X_test[cat_cols].astype(str)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_imp = SimpleImputer(strategy="median")
cat_imp = SimpleImputer(strategy="most_frequent")

Xtr_num = num_imp.fit_transform(X_train[num_cols]).astype(np.float32)
Xva_num = num_imp.transform(X_val[num_cols]).astype(np.float32)

Xtr_cat = cat_imp.fit_transform(X_train[cat_cols])
Xva_cat = cat_imp.transform(X_val[cat_cols])

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
Xtr_cat_ohe = ohe.fit_transform(Xtr_cat)
Xva_cat_ohe = ohe.transform(Xva_cat)

Xtr = hstack([csr_matrix(Xtr_num), Xtr_cat_ohe]).tocsr()
Xva = hstack([csr_matrix(Xva_num), Xva_cat_ohe]).tocsr()


In [ ]:
model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)
model.fit(Xtr, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.08, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=-1, num_parallel_tree=None, ...)

In [ ]:
pred_val = model.predict(Xva)
print("F1 (macro):", f1_score(y_val, pred_val, average="macro"))
print(classification_report(y_val, pred_val))

F1 (macro): 0.7797031372905778
              precision    recall  f1-score   support

           0       0.81      0.68      0.74      8914
           1       0.78      0.87      0.82     11290

    accuracy                           0.79     20204
   macro avg       0.79      0.78      0.78     20204
weighted avg       0.79      0.79      0.78     20204



In [ ]:
submission = pd.DataFrame()

Xte_num = num_imp.transform(X_test[num_cols]).astype(np.float32)
Xte_cat = cat_imp.transform(X_test[cat_cols])
Xte_cat_ohe = ohe.transform(Xte_cat)
X_test_final = hstack([csr_matrix(Xte_num), Xte_cat_ohe]).tocsr()

pred_test = model.predict(X_test_final)
submission = pd.DataFrame({"TripReason": le.inverse_transform(pred_test)})
submission


,TripReason
0,Int
1,Int
2,Work
3,Work
4,Work
...,...
43288,Work
43289,Work
43290,Work
43291,Work
